# Ramsey Calibration — CGCS / Triplet-Phase-Lock Starter Notebook

**Repository:** `quantum-hardware-company`  
**Directory:** `calibration/ramsey/`

This notebook initializes a Ramsey calibration workflow:

1. Generate synthetic Ramsey fringe data.
2. Fit a Ramsey decay/fringe model.
3. Extract detuning and dephasing proxy.
4. Analyze residual structure.
5. Compute a CGCS / Triplet-Phase-Lock metric using cosine similarity.
6. Save figures and a machine-readable JSON summary.

## Principle

Ramsey calibration measures phase evolution, detuning, and coherence decay.

Control software should follow measured phase behavior, not idealized assumptions.


## Ramsey model

We use the calibration primitive:

$$
P(t)=B + A e^{-t/T_2^*}\cos(\Delta t + \phi)
$$

where:

- $A$ is contrast / amplitude,
- $T_2^*$ is the Ramsey dephasing time,
- $\Delta$ is detuning / fringe angular frequency,
- $\phi$ is phase offset,
- $B$ is readout offset / baseline.


In [ ]:
import os
import json

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

np.random.seed(9423)

FIG_DIR = "figures"
RESULTS_DIR = "results"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
def ramsey_model(t, A, T2_star, Delta, phi, B):
    """Ramsey fringe model with exponential dephasing envelope."""
    return B + A * np.exp(-t / T2_star) * np.cos(Delta * t + phi)


def fit_model(model, t, y, p0, bounds=(-np.inf, np.inf)):
    """Fit model to data and return best-fit params + covariance."""
    params, cov = curve_fit(model, t, y, p0=p0, bounds=bounds, maxfev=30000)
    return params, cov


def residuals(model, t, y, params):
    """Observed minus fitted values."""
    return y - model(t, *params)


def phase_lock_metric(signal, fit):
    """Cosine similarity between observed signal and fitted model."""
    dot = np.dot(signal, fit)
    norm = np.linalg.norm(signal) * np.linalg.norm(fit)
    if norm == 0:
        return np.nan
    return dot / norm


def is_phase_locked(metric, threshold=1 / np.sqrt(2)):
    """CGCS threshold: cos(theta) >= 1/sqrt(2), equivalent to <= 45 degrees."""
    return metric >= threshold


def autocorrelation(x):
    """Simple non-normalized autocorrelation for residual structure checks."""
    x = x - np.mean(x)
    result = np.correlate(x, x, mode="full")
    return result[result.size // 2:]


## Generate synthetic calibration data

This cell stands in for experimental Ramsey readout data. Later, replace this block with a loader for real calibration data.


In [ ]:
# Time axis, arbitrary units
n_points = 260
t = np.linspace(0, 18, n_points)

# Ground-truth synthetic parameters
true_A = 0.42
true_T2_star = 9.5
true_Delta = 2.15
true_phi = 0.35
true_B = 0.50
true_params = [true_A, true_T2_star, true_Delta, true_phi, true_B]

# Noise model: measurement noise + slow structured drift component
measurement_noise = 0.035 * np.random.randn(n_points)
structured_drift = 0.012 * np.sin(0.22 * t + 0.8)

y_clean = ramsey_model(t, *true_params)
y_obs = y_clean + measurement_noise + structured_drift

# Probability clipping simulates bounded population readout
y_obs = np.clip(y_obs, 0, 1)

print("True parameters:")
print(f"A       = {true_A:.4f}")
print(f"T2*     = {true_T2_star:.4f}")
print(f"Delta   = {true_Delta:.4f}")
print(f"phi     = {true_phi:.4f}")
print(f"B       = {true_B:.4f}")


## Fit Ramsey model


In [ ]:
initial_guess = [0.35, 8.0, 2.0, 0.0, 0.50]

# Bounds keep fit physically interpretable for this starter workflow:
# A >= 0, T2* > 0, B in [0, 1]
lower_bounds = [0.0, 0.5, 0.0, -2 * np.pi, 0.0]
upper_bounds = [1.0, 100.0, 10.0, 2 * np.pi, 1.0]

params, cov = fit_model(
    ramsey_model,
    t,
    y_obs,
    p0=initial_guess,
    bounds=(lower_bounds, upper_bounds),
)

A_fit, T2_star_fit, Delta_fit, phi_fit, B_fit = params
param_std = np.sqrt(np.diag(cov))
y_fit = ramsey_model(t, *params)

print("Fit parameters:")
print(f"A       = {A_fit:.4f} ± {param_std[0]:.4f}")
print(f"T2*     = {T2_star_fit:.4f} ± {param_std[1]:.4f}")
print(f"Delta   = {Delta_fit:.4f} ± {param_std[2]:.4f}")
print(f"phi     = {phi_fit:.4f} ± {param_std[3]:.4f}")
print(f"B       = {B_fit:.4f} ± {param_std[4]:.4f}")


## Plot data and fit


In [ ]:
plt.figure(figsize=(9, 5))
plt.scatter(t, y_obs, s=18, label="observed Ramsey data")
plt.plot(t, y_fit, linewidth=2, label="Ramsey fit")
plt.xlabel("free evolution time")
plt.ylabel("excited-state probability")
plt.title("Ramsey calibration: measured fringes and fitted model")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/ramsey_01_fit.png", dpi=300)
plt.savefig(f"{FIG_DIR}/ramsey_01_fit.pdf")

plt.show()


## Envelope visualization

The fitted exponential envelope helps make the dephasing scale visually explicit.


In [ ]:
upper_envelope = B_fit + A_fit * np.exp(-t / T2_star_fit)
lower_envelope = B_fit - A_fit * np.exp(-t / T2_star_fit)

plt.figure(figsize=(9, 5))
plt.scatter(t, y_obs, s=14, label="observed Ramsey data")
plt.plot(t, y_fit, linewidth=2, label="Ramsey fit")
plt.plot(t, upper_envelope, linestyle="--", linewidth=1, label="fitted envelope (upper)")
plt.plot(t, lower_envelope, linestyle="--", linewidth=1, label="fitted envelope (lower)")
plt.xlabel("free evolution time")
plt.ylabel("excited-state probability")
plt.title("Ramsey calibration: fitted dephasing envelope")
plt.legend()
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/ramsey_02_envelope.png", dpi=300)
plt.savefig(f"{FIG_DIR}/ramsey_02_envelope.pdf")

plt.show()


## Residual analysis

Residuals are treated as measurable structure unless demonstrated random.


In [ ]:
res = residuals(ramsey_model, t, y_obs, params)
rmse = np.sqrt(np.mean(res**2))
mean_res = np.mean(res)
std_res = np.std(res)

print(f"Residual mean = {mean_res:.6f}")
print(f"Residual std  = {std_res:.6f}")
print(f"Fit RMSE      = {rmse:.6f}")


In [ ]:
plt.figure(figsize=(9, 4))
plt.axhline(0, linewidth=1)
plt.plot(t, res, marker="o", markersize=3, linewidth=1)
plt.xlabel("free evolution time")
plt.ylabel("residual: observed - fit")
plt.title("Ramsey calibration residuals")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/ramsey_03_residuals.png", dpi=300)
plt.savefig(f"{FIG_DIR}/ramsey_03_residuals.pdf")

plt.show()


## Residual autocorrelation

This diagnostic checks whether residuals contain structure over lag rather than behaving like independent random noise.


In [ ]:
ac = autocorrelation(res)

plt.figure(figsize=(7, 4))
plt.plot(ac[:60])
plt.title("Ramsey residual autocorrelation (structure check)")
plt.xlabel("lag")
plt.ylabel("correlation")
plt.tight_layout()

plt.savefig(f"{FIG_DIR}/ramsey_04_residual_autocorr.png", dpi=300)
plt.savefig(f"{FIG_DIR}/ramsey_04_residual_autocorr.pdf")

plt.show()


## CGCS / Triplet-Phase-Lock metric

We compute cosine similarity between observed signal and fitted model.

The working CGCS threshold is:

$$
\cos\theta \geq \frac{1}{\sqrt{1^2+1^2}}
$$

This is the 45° constraint gate.


In [ ]:
metric = phase_lock_metric(y_obs, y_fit)
threshold = 1 / np.sqrt(2)
locked = is_phase_locked(metric, threshold=threshold)
angle_deg = np.degrees(np.arccos(np.clip(metric, -1, 1)))

print(f"CGCS phase-lock metric = {metric:.6f}")
print(f"CGCS threshold         = {threshold:.6f}")
print(f"Angle                  = {angle_deg:.3f} degrees")
print(f"Phase locked?          = {locked}")


## CGCS interpretation

A phase-lock angle far below 45° indicates that the measured Ramsey signal remains inside the constraint region. In this synthetic calibration run, strong alignment means the fitted dephasing/fringe model explains the observed signal before downstream noise-mitigation or control-stack adjustments.


In [ ]:
plt.figure(figsize=(7, 4))
labels = ["fit alignment", "45° threshold"]
values = [metric, threshold]

bars = plt.bar(labels, values)
plt.axhline(threshold, linewidth=1, linestyle="--")
plt.ylim(0, 1.05)
plt.ylabel("cosine similarity")
plt.title("CGCS phase-lock check")

for bar, value in zip(bars, values):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.015,
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()

plt.savefig(f"{FIG_DIR}/ramsey_05_phase_lock.png", dpi=300)
plt.savefig(f"{FIG_DIR}/ramsey_05_phase_lock.pdf")

plt.show()


## Calibration summary


In [ ]:
summary = {
    "A_fit": float(A_fit),
    "T2_star_fit": float(T2_star_fit),
    "Delta_fit": float(Delta_fit),
    "phi_fit": float(phi_fit),
    "B_fit": float(B_fit),
    "residual_rmse": float(rmse),
    "phase_lock_metric": float(metric),
    "phase_lock_threshold": float(threshold),
    "phase_lock_angle_degrees": float(angle_deg),
    "phase_locked": bool(locked),
}

summary


## Save calibration summary


In [ ]:
with open(f"{RESULTS_DIR}/ramsey_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved {RESULTS_DIR}/ramsey_summary.json")


## Zip outputs for download from Colab

Run this cell after all figures/results have been generated.


In [ ]:
!zip -r ramsey_outputs.zip figures results


## Next steps

This notebook connects directly to:

- `calibration/rabi/` for amplitude/frequency control,
- `calibration/rb/` for gate-fidelity decay,
- `calibration/drift/` for stability over time,
- `noise-mitigation/` for structured dephasing and detuning drift,
- `control-stack/` for detuning-aware pulse tuning.

Guiding rule:

> Start where physics shows up.
